# 🎯 Qwen LoRA Fine-Tune (ChatML format) — Public Template

**License:** MIT  
**Base model:** Qwen3.5-4B (configurable)  
**Format:** ChatML (`<|im_start|>...<|im_end|>`)  
**Output:** LoRA adapter → merged model → Q6_K GGUF → HuggingFace Hub

---

## ⚙️ How to Use

1. **Edit Cell `[CONFIG]`** below — set your dataset path, HF repo, system prompt, hyperparameters.
2. **Add a Kaggle Secret** named `HF_TOKEN` (Add-ons → Secrets) for HuggingFace push.
3. **Run All** — the notebook trains LoRA on dual T4 (~3h for 1000 steps), merges, exports Q6_K GGUF, and pushes to your HF repo.

## 📦 Dataset Format

Expected JSONL with two fields per line:
```json
{"instruction": "...", "response": "..."}
```

## 🆚 ChatML vs Alpaca

This notebook uses **ChatML** training format. For the Alpaca variant (same code, different formatter), see `qwen-lora-finetune-alpaca.ipynb` in the same folder.

---

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [CONFIG] — Edit ONLY this cell to customize the training run
# ══════════════════════════════════════════════════════════════════════

# ── Model & Dataset ──────────────────────────────────────────────────
BASE_MODEL    = "Qwen/Qwen3.5-4B"           # any HuggingFace causal-LM repo
DATASET_PATH  = "/kaggle/input/your-dataset/your_data.jsonl"
                                              # JSONL with {instruction, response}

# ── HuggingFace push (set HF_TOKEN secret in Kaggle Add-ons → Secrets) ──
HF_REPO       = "your-username/your-model-name"   # repo will be created
HF_PRIVATE    = True

# ── System prompt (PLACEHOLDER — replace with your task instructions) ──
SYSTEM_PROMPT = """<Replace with your task instructions>

Example:
  You are a helpful assistant. When the user asks a question, respond
  with a clear, concise, accurate answer.
"""

# ── Training hyperparameters ─────────────────────────────────────────
MAX_STEPS     = 1000                          # total optimization steps
LR            = 5e-5                          # learning rate
WARMUP_STEPS  = 50
LORA_R        = 16                            # LoRA rank (8/16/32 typical)
LORA_ALPHA    = 32                            # alpha = 2*r is a common default
BATCH         = 1                             # per-device batch size
GRAD_ACCUM    = 4                             # effective batch = BATCH*GPUs*GRAD_ACCUM
MAX_SEQ_LEN   = 2048
SEED          = 3407
LOGGING_STEPS = 10
SAVE_STRATEGY = "steps"
SAVE_STEPS    = 200

# ── Output ───────────────────────────────────────────────────────────
OUTPUT_DIR    = "./lora_adapter"
MERGED_DIR    = "./merged_model"
GGUF_OUT      = "model_Q6_K.gguf"

print(f"✓ Config loaded: {BASE_MODEL} → {HF_REPO} ({MAX_STEPS} steps, lr={LR}, r={LORA_R})")


In [ ]:
# ============================================================
# [1/12] GPU Isolation + Global Timer
# ============================================================
import os, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
NOTEBOOK_START = time.time()
print("✅ [1/12] GPU environment variables set")
print(f"   CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"   Start time: {time.strftime('%H:%M:%S')}")


In [ ]:
# ============================================================
# [2/12] Library Installation (verbose, no capture)
# ============================================================
import subprocess, sys
print("="*60)
print("[2/12] Installing libraries...")
print("="*60)
# Qwen 3.5 needs transformers >= 4.51
pkgs = [
    "transformers accelerate --upgrade",
    "datasets bitsandbytes",
    "peft",
    "trl",
]
for pkg in pkgs:
    cmd = f"{sys.executable} -m pip install -q {pkg}"
    print(f"   📦 {pkg.split()[0]}...")
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   ❌ ERROR: {r.stderr[-200:]}")
    else:
        print(f"   ✅ OK")

# Version check
import transformers, trl, peft, torch
print(f"\n📋 INSTALLED VERSIONS:")
print(f"   transformers: {transformers.__version__}")
print(f"   trl:          {trl.__version__}")
print(f"   peft:         {peft.__version__}")
print(f"   torch:        {torch.__version__}")
print(f"   CUDA:         {torch.version.cuda}")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [3/12] System Status (GPU + Disk)
# ============================================================
import torch, shutil, subprocess
print("="*60)
print("[3/12] System Status")
print("="*60)

# GPU
print(f"\n🔧 CUDA: {torch.cuda.is_available()}")
print(f"🔧 GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"   GPU {i}: {props.name} | {props.total_memory/1024**3:.1f} GB VRAM")

# Disk
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\n💾 DISK STATUS:")
print(f"   Total: {total/1024**3:.1f} GB")
print(f"   Used:  {used/1024**3:.1f} GB")
print(f"   Free:  {free/1024**3:.1f} GB")
if free/1024**3 < 15:
    print(f"   ⚠️ WARNING: Low disk space! GGUF conversion needs ≥15 GB.")
else:
    print(f"   ✅ Disk space OK")

# nvidia-smi
r = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv,noheader"], capture_output=True, text=True)
print(f"\n📊 nvidia-smi:\n{r.stdout}")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [4/12] Model & Tokenizer Loading
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
print("="*60)
print("[4/12] Loading model...")
print("="*60)

print(f"📥 Model: {BASE_MODEL}")
print(f"📏 Max sequence length: {MAX_SEQ_LEN}")
print(f"🔀 Strategy: FP16 + device_map=auto")
print(f"   Loading...")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
print(f"✅ Model loaded!")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✅ Tokenizer loaded!")

# VRAM report
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i)/1024**3
    total = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f"   GPU {i}: {alloc:.2f} / {total:.1f} GB ({alloc/total*100:.0f}%)")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [5/12] LoRA Adapter
# ============================================================
from peft import LoraConfig, get_peft_model
print("="*60)
print("[5/12] Adding LoRA adapter...")
print("="*60)

model.gradient_checkpointing_enable()
print("✅ Gradient checkpointing enabled")

TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n📋 LoRA SETTINGS:")
print(f"   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"   scaling factor (alpha/r) = {LORA_ALPHA/LORA_R}")
print(f"   target modules: {TARGET_MODULES}")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# [6/12] Dataset Loading + ChatML Formatting
# ══════════════════════════════════════════════════════════════════════
import os
from datasets import load_dataset

# Auto-discover dataset if exact path missing (Kaggle path may vary)
if not os.path.exists(DATASET_PATH):
    print(f"⚠ {DATASET_PATH} not found — searching /kaggle/input/...")
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.jsonl'):
                DATASET_PATH = os.path.join(root, f)
                print(f"  ✓ Found: {DATASET_PATH}")
                break
        if os.path.exists(DATASET_PATH):
            break

dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f"✓ Dataset loaded: {len(dataset)} examples")
print(f"  Fields: {list(dataset.column_names)}")

# ── ChatML Format ────────────────────────────────────────────────────
# <|im_start|>system\n{sys}<|im_end|>\n<|im_start|>user\n{i}<|im_end|>\n<|im_start|>assistant\n{r}<|im_end|>
def format_prompts(examples):
    texts = []
    for inst, resp in zip(examples['instruction'], examples['response']):
        chatml = (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{inst}<|im_end|>\n"
            f"<|im_start|>assistant\n{resp}<|im_end|>"
        )
        texts.append(chatml)
    return {'text': texts}

formatted_dataset = dataset.map(
    format_prompts, batched=True, remove_columns=dataset.column_names
)
print(f"✓ Formatted {len(formatted_dataset)} examples (ChatML)")
print(f"\nSample (first 300 chars):\n{formatted_dataset[0]['text'][:300]}...")


In [ ]:
# ============================================================
# [7/12] Pre-Training Cache + Disk Check
# ============================================================
import torch, shutil
print("="*60)
print("[7/12] Clearing cache...")
print("="*60)
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i)/1024**3
    total = torch.cuda.get_device_properties(i).total_memory/1024**3
    free = total - alloc
    print(f"   GPU {i}: {alloc:.2f} / {total:.1f} GB — Free: {free:.2f} GB")

total, used, free = shutil.disk_usage("/kaggle/working")
print(f"\n💾 Disk: {used/1024**3:.1f}/{total/1024**3:.1f} GB used, {free/1024**3:.1f} GB free")
print(f"✅ Ready!")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [8/12] TRAINING (verbose monitoring)
# ============================================================
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
import torch, shutil
print("="*60)
print("[8/12] TRAINING STARTING")
print("="*60)

# ─── Verbose Monitoring Callback ─────────────────────────────────────
class VerboseMonitor(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            step = state.global_step
            loss = logs["loss"]
            lr = logs.get("learning_rate", 0)
            elapsed = time.time() - NOTEBOOK_START
            vram_0 = torch.cuda.memory_allocated(0)/1024**3
            vram_1 = torch.cuda.memory_allocated(1)/1024**3 if torch.cuda.device_count()>1 else 0
            _, _, disk_free = shutil.disk_usage("/kaggle/working")
            disk_gb = disk_free/1024**3
            if step > 0:
                sec_per_step = elapsed / step
                remaining = (args.max_steps - step) * sec_per_step
                eta_h = remaining / 3600
            else:
                eta_h = 0
            pct = step / args.max_steps * 100
            bar = "█" * int(pct/5) + "░" * (20-int(pct/5))
            print(f"  [{bar}] {pct:.0f}% | Step {step}/{args.max_steps} | Loss: {loss:.4f} | LR: {lr:.2e} | VRAM: {vram_0:.1f}+{vram_1:.1f}GB | Disk: {disk_gb:.1f}GB | ETA: {eta_h:.1f}h")

    def on_train_begin(self, args, state, control, **kwargs):
        print("\n🚀 TRAINING STARTED!")
        print(f"   max_steps:     {args.max_steps}")
        print(f"   save_strategy: {args.save_strategy}")
        print(f"   logging_steps: {args.logging_steps}")
        print(f"   optimizer:     {args.optim}")
        print(f"   fp16:          {args.fp16}")
        print(f"   seed:          {args.seed}")

    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n🏁 TRAINING DONE! Final step: {state.global_step}")

print(f"\n⚙️ TRAINING CONFIG (from [CONFIG] cell):")
print(f"   per_device_batch_size: {BATCH}")
print(f"   gradient_accumulation: {GRAD_ACCUM}")
print(f"   max_steps:             {MAX_STEPS}")
print(f"   learning_rate:         {LR}")
print(f"   warmup_steps:          {WARMUP_STEPS}")
print(f"   max_seq_length:        {MAX_SEQ_LEN}")
print(f"   save_strategy:         {SAVE_STRATEGY}")
print(f"   save_steps:            {SAVE_STEPS}")
print(f"   logging_steps:         {LOGGING_STEPS}")
print(f"   optimizer:             adamw_8bit")
print(f"   seed:                  {SEED}")
print("="*60)

train_start = time.time()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LEN,
        packing=False,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        fp16=True,
        logging_steps=LOGGING_STEPS,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=SEED,
        output_dir="outputs",
        save_strategy=SAVE_STRATEGY,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="none",
    ),
    callbacks=[VerboseMonitor()],
)
print("✅ Trainer ready, starting training...")
print("="*60)

trainer_stats = trainer.train()

train_duration = time.time() - train_start
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print(f"   Final loss:      {trainer_stats.training_loss:.4f}")
print(f"   Total steps:     {trainer_stats.global_step}")
print(f"   Training time:   {train_duration/3600:.1f} h ({train_duration:.0f}s)")
print(f"   Per step:        {train_duration/trainer_stats.global_step:.1f}s")

_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"   Disk free:       {disk_free/1024**3:.1f} GB")
print(f"⏱️ Total elapsed: {time.time()-NOTEBOOK_START:.1f}s")
print("="*60)


In [ ]:
# ============================================================
# [9/12] Save LoRA Adapter + Free Memory
# ============================================================
import gc, shutil
print("="*60)
print("[9/12] Saving LoRA adapter...")
print("="*60)

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ LoRA adapter saved: {OUTPUT_DIR}")

if 'model' in dir(): del model
if 'trainer' in dir(): del trainer
torch.cuda.empty_cache()
gc.collect()

print(f"📊 GPU 0 VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"💾 Disk free: {disk_free/1024**3:.1f} GB")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [10/12] Safe CPU Merge
# ============================================================
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, gc, shutil
print("="*60)
print("[10/12] Merging on CPU")
print("="*60)

print("📥 Loading base model on CPU (2-5 min)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
print("✅ Base model loaded!")

print("🔗 Merging LoRA...")
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()
print("✅ Merge complete!")

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✅ Merged model saved: {MERGED_DIR}")

del base_model, merged_model
gc.collect()
_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"💾 Disk free: {disk_free/1024**3:.1f} GB")
if disk_free/1024**3 < 10:
    print("⚠️ WARNING: Low disk! Removing LoRA adapter...")
    import shutil as sh
    sh.rmtree(OUTPUT_DIR, ignore_errors=True)
    sh.rmtree("outputs", ignore_errors=True)
    _, _, disk_free = shutil.disk_usage("/kaggle/working")
    print(f"💾 After cleanup: {disk_free/1024**3:.1f} GB free")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [11/12] GGUF Conversion (F16 → Q6_K)
# ============================================================
import subprocess, sys, os, shutil
print("="*60)
print("[11/12] GGUF Conversion")
print("="*60)

_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"💾 Disk free: {disk_free/1024**3:.1f} GB")

# llama.cpp
print("[1/4] Cloning llama.cpp...")
if not os.path.exists("/kaggle/working/llama_cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama_cpp
else:
    print("   Already present")

print("[2/4] Installing GGUF dependencies...")
!pip install -q --no-deps gguf
!pip install -q protobuf pyyaml sentencepiece

# Workaround for llama.cpp tokenizer hash mismatch — re-download originals
print("[2.5/4] Downloading original tokenizer files (llama.cpp hash workaround)...")
from huggingface_hub import hf_hub_download
for f in ['tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']:
    try:
        hf_hub_download(repo_id=BASE_MODEL, filename=f, local_dir=MERGED_DIR)
    except Exception:
        pass

print("[3/4] Converting to F16 GGUF...")
r = subprocess.run(
    [sys.executable, "/kaggle/working/llama_cpp/convert_hf_to_gguf.py",
     MERGED_DIR, "--outfile", "model_f16.gguf", "--outtype", "f16"],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else "")
if r.returncode != 0:
    print(f"❌ ERROR: {r.stderr[-500:]}")
    raise RuntimeError("GGUF conversion failed!")
print("✅ F16 GGUF OK!")

# Free disk: merged_model no longer needed
print(f"🗑️ Removing {MERGED_DIR} (disk savings)...")
import shutil as sh
sh.rmtree(MERGED_DIR, ignore_errors=True)
_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"💾 Disk free: {disk_free/1024**3:.1f} GB")

print("[4/4] Q6_K quantization...")
quantize_bin = "/kaggle/working/llama_cpp/llama-quantize"
if not os.path.exists(quantize_bin):
    r2 = subprocess.run(["cmake","--version"], capture_output=True, text=True)
    if r2.returncode == 0:
        !cd /kaggle/working/llama_cpp && cmake -B build -DGGML_CUDA=OFF 2>/dev/null && cmake --build build --target llama-quantize -j$(nproc) 2>/dev/null
        for p in ["/kaggle/working/llama_cpp/build/bin/llama-quantize", "/kaggle/working/llama_cpp/build/llama-quantize"]:
            if os.path.exists(p):
                quantize_bin = p
                break
    else:
        !cd /kaggle/working/llama_cpp && make -j$(nproc) llama-quantize 2>/dev/null

if os.path.exists(quantize_bin):
    !{quantize_bin} model_f16.gguf {GGUF_OUT} Q6_K
    os.remove("model_f16.gguf")
    print("✅ Q6_K OK! F16 deleted.")
else:
    print("⚠️ quantize binary missing, keeping F16")
    os.rename("model_f16.gguf", GGUF_OUT)

!ls -lh *.gguf
_, _, disk_free = shutil.disk_usage("/kaggle/working")
print(f"💾 Disk free: {disk_free/1024**3:.1f} GB")
print(f"⏱️ Elapsed: {time.time()-NOTEBOOK_START:.1f}s")


In [ ]:
# ============================================================
# [12/12] HuggingFace Upload
# ============================================================
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
import os, time

print("="*60)
print("[12/12] HuggingFace Upload")
print("="*60)

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    repo_id = HF_REPO
    visibility = "PRIVATE" if HF_PRIVATE else "PUBLIC"
    print(f"📦 Repo: {repo_id} ({visibility})")

    api = HfApi()
    api.create_repo(repo_id=repo_id, token=hf_token, exist_ok=True, private=HF_PRIVATE)

    if os.path.exists(GGUF_OUT):
        fsize = os.path.getsize(GGUF_OUT)/1024**3
        print(f"📤 Uploading: {GGUF_OUT} ({fsize:.2f} GB)")

        api.upload_file(
            path_or_fileobj=GGUF_OUT,
            path_in_repo=GGUF_OUT,
            repo_id=repo_id,
            token=hf_token,
        )

        try:
            total_time = time.time() - NOTEBOOK_START
        except NameError:
            total_time = 0

        print(f"\n{'='*60}")
        print(f"🎉 DONE!")
        print(f"   Model: https://huggingface.co/{repo_id}")
        if total_time > 0:
            print(f"   Total time: {total_time/3600:.1f} h ({total_time:.0f}s)")
        print(f"{'='*60}")
    else:
        print(f"❌ ERROR: {GGUF_OUT} not found. Make sure step [11/12] completed successfully.")

except Exception as e:
    print(f"❌ Upload error: {e}")
    print("💡 Make sure you've added a Kaggle Secret named 'HF_TOKEN' in Add-ons → Secrets.")
